# Lorde Forge · Kaggle runbook (Phase 0)

Runs the whole pipeline on a **Kaggle 2×T4** notebook (Internet on, GPU T4 x2). Three sessions:

| Session | Steps | Envs | Time |
|---|---|---|---|
| A · data | 0-3: serve teacher/student, build pool, teacher labels, splits, golden set | serve, data | 2-3 h |
| B · train | 5: LoRA SFT with AlignTune, merge | train | 2-3 h |
| C · evaluate | 6-9: judge eval, CuratorKIT plug-in runs, downstream test, bench, publish | serve, data, train | 3-4 h |

Step 4 (hand labels) runs on your laptop: `python forge/04_label_tool.py --share`.

**Secrets** (Add-ons → Secrets): `HF_TOKEN` (read access to the gated Cohere models; write scope for publishing),
`COHERE_API_KEY` (32B calibration slice, optional). Accept the licences on the Tiny Aya Global, Tiny Aya Base and
Aya Expanse 8B model pages first.

`data/` is synced to a **private** HF dataset (`<you>/lorde-work`) at the end of each session, so a killed
session loses nothing: every step resumes from its cache.

In [ ]:
# ---- configure ----
HF_USER = "your-hf-username"          # <- change
REPO = "https://github.com/linoxbt/Lorde.git"
BRANCH = "main"                        # or "ccr-e4bea25f-7bzaqy" until it is merged
WORK_REPO = f"{HF_USER}/lorde-work"   # private dataset used as a session-to-session cache

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
try:
    os.environ["COHERE_API_KEY"] = s.get_secret("COHERE_API_KEY")
except Exception:
    print("no COHERE_API_KEY: the 32B calibration slice will be skipped")
os.environ["VLLM_API_KEY"] = "lorde-local"
os.environ["LITELLM_LOG"] = "ERROR"

In [ ]:
%%bash -s "$REPO" "$BRANCH"
cd /kaggle/working
[ -d Lorde ] || git clone -b "$2" "$1" Lorde
cd Lorde && git pull -q && git log --oneline | head -3
pip install -q uv

In [ ]:
%%bash
# Three separate environments (spec 4.1): installing AlignTune and vLLM together is the fastest way to lose a day.
cd /kaggle/working/Lorde
for env in serve data train; do
  [ -x envs/$env/bin/python ] || uv venv -q -p 3.11 envs/$env
done
uv pip install -q --python envs/serve/bin/python -r requirements/forge-serve.txt
uv pip install -q --python envs/data/bin/python -r requirements/forge-data.txt
uv pip install -q --python envs/train/bin/python -r requirements/forge-train.txt
envs/serve/bin/python -c "import vllm; print('vllm', vllm.__version__)"
envs/data/bin/python -c "import curatorkit, auditkit; print('curatorkit', curatorkit.__version__)"
envs/train/bin/python -c "import aligntune, trl; print('trl', trl.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Session-to-session cache: pull data/, runs/ (adapters) and results/ from the private work repo.
import os

from huggingface_hub import HfApi, snapshot_download

os.chdir("/kaggle/working/Lorde")
api = HfApi()
api.create_repo(WORK_REPO, repo_type="dataset", private=True, exist_ok=True)
snapshot_download(WORK_REPO, repo_type="dataset", local_dir=".", allow_patterns=["data/**", "results/**", "runs/**/merged/**", "runs/**/train_meta.json", "bench/**"])

def sync(patterns=("data/**", "results/**", "bench/**")):
    for p in patterns:
        base = p.split("/")[0]
        if os.path.exists(base):
            api.upload_folder(repo_id=WORK_REPO, repo_type="dataset", folder_path=base, path_in_repo=base,
                              ignore_patterns=["*.tmp", "_input/**"])
    print("synced", patterns)

## Session A · data (steps 0-3)

**Smoke-test vLLM on T4 first.** T4 is compute capability 7.5 (no bf16): `00_serve.sh` switches to fp16 by itself.
If the teacher fails to start, read `logs/vllm_teacher.log`; `VLLM_EXTRA_ARGS="--enforce-eager"` or
`GPU_UTIL=0.85` are the usual fixes.

In [ ]:
%%bash
cd /kaggle/working/Lorde && export PATH=$PWD/envs/serve/bin:$PATH
bash forge/00_serve.sh teacher
envs/data/bin/python forge/01_build_pool.py --phase teacher --teacher-model openai/teacher --teacher-base http://localhost:8000/v1

In [ ]:
sync()

In [ ]:
%%bash
cd /kaggle/working/Lorde && export PATH=$PWD/envs/serve/bin:$PATH
bash forge/00_serve.sh stop && CUDA_VISIBLE_DEVICES=0 bash forge/00_serve.sh student
envs/data/bin/python forge/01_build_pool.py --phase student --student-model openai/student --student-base http://localhost:8001/v1
envs/data/bin/python forge/01_build_pool.py --phase assemble

In [ ]:
%%bash
cd /kaggle/working/Lorde && export PATH=$PWD/envs/serve/bin:$PATH
bash forge/00_serve.sh stop && bash forge/00_serve.sh teacher
envs/data/bin/python forge/02_teacher_label.py --model openai/teacher --base http://localhost:8000/v1 --workers 32
cat results/labels_summary.json

In [ ]:
%%bash
# Exit check: parse-OK >= 95% and score bins span low/mid/high (printed above). Then splits and the golden set.
cd /kaggle/working/Lorde
envs/data/bin/python forge/03_make_splits.py --human-langs en,fr,es   # languages you can read
envs/data/bin/python bench/build_sets.py

In [ ]:
%%bash
# Calibration: Aya Expanse 32B through the Cohere API on golden-300 (300 calls; trial keys allow 1,000/month).
cd /kaggle/working/Lorde
if [ -n "${COHERE_API_KEY:-}" ]; then
  envs/data/bin/python forge/02_teacher_label.py --model c4ai-aya-expanse-32b \
    --base https://api.cohere.ai/compatibility/v1 --api-key-env COHERE_API_KEY \
    --sample-from bench/data/golden-300.jsonl --max-calls 300 --workers 4 --out data/labels_32b.jsonl
  envs/data/bin/python bench/build_sets.py   # attach the 32B labels to golden-300
fi

In [ ]:
%%bash
cd /kaggle/working/Lorde && export PATH=$PWD/envs/serve/bin:$PATH && bash forge/00_serve.sh stop

In [ ]:
sync()

## Step 4 · human labels (laptop)

```bash
git clone https://github.com/linoxbt/Lorde && cd Lorde && pip install -r requirements/forge-data.txt
huggingface-cli download <you>/lorde-work --repo-type dataset --include "data/*" --local-dir .
python forge/04_label_tool.py --labeler <name> --share     # 200 items, 1-2 h
huggingface-cli upload <you>/lorde-work data/human_labels.jsonl data/human_labels.jsonl --repo-type dataset
```

## Session B · train (step 5)

Smoke-test with `--max-samples 64 --epochs 1` first. If AlignTune misbehaves with Tiny Aya, use `--engine trl`.

In [ ]:
%%bash
cd /kaggle/working/Lorde
CUDA_VISIBLE_DEVICES=0 envs/train/bin/python forge/05_train_sft.py --max-samples 64 --epochs 1 --run smoke

In [ ]:
%%bash
cd /kaggle/working/Lorde
CUDA_VISIBLE_DEVICES=0 envs/train/bin/python forge/05_train_sft.py            # add --engine trl as fallback
cat runs/lorde-v1/train_meta.json

In [ ]:
# Save the merged model to the work repo so Session C can load it.
api.upload_folder(repo_id=WORK_REPO, repo_type="dataset", folder_path="runs/lorde-v1",
                  path_in_repo="runs/lorde-v1", allow_patterns=["merged/**", "train_meta.json"])

## Session C · evaluate (steps 6-9)

In [ ]:
%%bash
cd /kaggle/working/Lorde && export PATH=$PWD/envs/serve/bin:$PATH
bash forge/00_serve.sh stop
CUDA_VISIBLE_DEVICES=0 bash forge/00_serve.sh student
CUDA_VISIBLE_DEVICES=1 bash forge/00_serve.sh lorde
envs/data/bin/python forge/06_judge_eval.py \
  --judge base=openai/student@http://localhost:8001/v1 \
  --judge lorde=openai/lorde@http://localhost:8002/v1 \
  --gpu-hourly base=0.50 --gpu-hourly lorde=0.50 --structured
envs/data/bin/python bench/run_bench.py --judge lorde=openai/lorde@http://localhost:8002/v1 || true
envs/data/bin/python bench/run_bench.py --judge lorde=openai/lorde@http://localhost:8002/v1 --structured || true
envs/data/bin/python forge/07_curatorkit_plugin.py \
  --judge base=openai/student@http://localhost:8001/v1 \
  --judge lorde=openai/lorde@http://localhost:8002/v1
envs/data/bin/python forge/07_curatorkit_plugin.py --judge lorde=openai/lorde@http://localhost:8002/v1 --structured

In [ ]:
%%bash
cd /kaggle/working/Lorde && export PATH=$PWD/envs/serve/bin:$PATH
bash forge/00_serve.sh stop && bash forge/00_serve.sh teacher
envs/data/bin/python forge/07_curatorkit_plugin.py --judge teacher=openai/teacher@http://localhost:8000/v1
bash forge/00_serve.sh stop

In [ ]:
sync()

In [ ]:
%%bash
# Downstream: three small SFTs (ungated / teacher-gated / Lorde-gated), answers, AuditKIT scores. 1.5-2.5 h.
cd /kaggle/working/Lorde
CUDA_VISIBLE_DEVICES=0 envs/train/bin/python forge/08_downstream.py all --sets ungated,teacher,lorde --include-base
cat results/downstream.md

In [ ]:
sync()

In [ ]:
%%bash -s "$HF_USER"
cd /kaggle/working/Lorde
envs/data/bin/python forge/09_publish.py --user "$1" --space          # private: review on the Hub first
# envs/data/bin/python forge/09_publish.py --user "$1" --space --public   # then make it public (un-gated)